# EDA of the sample videos — Destroyerrrrs

Runs from a clean clone: it reads `docs/assets/data.json`, which the pipeline tooling produced from the four sample videos (per-10 s object counts by class, brightness and motion curves, signal-phase runs, our detected events and the team's labels). The heavier artefacts (motion heatmaps, trajectory plots, flow field, alignment) are the images under `docs/assets/`, produced by the scripts in `tools/`.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
D = json.load(open(ROOT / 'docs/assets/data.json'))
VIDEOS = list(D)
for v in VIDEOS:
    d = D[v]
    print(f"{v}: {d['duration']:.1f} s @ {d['fps']} fps, {d['n_frames']} frames, mean brightness {d['mean_brightness']}, {d['unique_tracks']} tracks")

## Lighting
Two daytime clips, one evening clip, one dusk clip with headlights. Detection had to hold across this range, which is why we kept a medium detector at 1280 px rather than a nano model.

In [ ]:
plt.figure(figsize=(10,3.5))
for v in VIDEOS:
    b = np.array(D[v]['brightness']); plt.plot(b[:,0], b[:,1], label=v, lw=1.2)
plt.xlabel('s'); plt.ylabel('mean frame intensity'); plt.legend(); plt.title('Brightness over time'); plt.show()

## Road users over time
Distinct tracked objects per 10-second window. Pedestrians outnumber vehicles in every clip; the sawtooth in cars follows the ~77 s signal cycle (red ≈ 45 s, green ≈ 32 s).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for ax, v in zip(axes.ravel(), VIDEOS):
    c = D[v]['counts_per_10s']; t = np.arange(len(c['person'])) * 10
    for k in ('person', 'car', 'bus', 'truck'):
        if k in c: ax.plot(t, c[k], label=k, lw=1.2)
    ax.set_title(v); ax.set_xlabel('s')
axes[0,0].legend(); axes[0,0].set_ylabel('objects / 10 s'); plt.tight_layout(); plt.show()

## Signal phases
Read from lamp pixels of the two visible signal heads (see `src/signal.py`). Amber is only shown by the far-pole head; the near-pole head goes red ~6 s earlier, which matters for the `red_light` rule.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
col = {'red': '#ff4d4d', 'green': '#51cf66', 'amber': '#ffd43b', 'unknown': '#555'}
for i, v in enumerate(VIDEOS):
    for s, a, b in D[v]['signal']:
        ax.barh(i, b - a, left=a, color=col.get(s, '#555'), height=0.6)
ax.set_yticks(range(len(VIDEOS))); ax.set_yticklabels(VIDEOS); ax.set_xlabel('s'); ax.set_title('Signal state per video'); plt.tight_layout(); plt.show()

## Events: detected vs labelled
Our detections (top row per video) against the team's labels (bottom row), by class. Same-class label overlaps are merged as the organizers do for simultaneous events.

In [ ]:
CLASSES = ['red_light','stop_line','stopped_vehicle','jaywalking','failure_to_yield','wrong_way','solid_line_crossing','congestion','illegal_turn','illegal_u_turn']
cmap = plt.get_cmap('tab10'); C = {c: cmap(i) for i, c in enumerate(CLASSES)}
fig, axes = plt.subplots(len(VIDEOS), 1, figsize=(12, 7), sharex=False)
for ax, v in zip(axes, VIDEOS):
    for s, e, l in D[v]['events']: ax.barh(1, e - s, left=s, color=C.get(l, '#999'), height=0.7)
    for s, e, l in D[v]['labels']: ax.barh(0, e - s, left=s, color=C.get(l, '#999'), height=0.7, alpha=0.6)
    ax.set_yticks([0, 1]); ax.set_yticklabels(['labels', 'detected']); ax.set_title(v, loc='left', fontsize=10); ax.set_xlim(0, D[v]['duration'])
handles = [plt.Rectangle((0,0),1,1,color=C[c]) for c in CLASSES]; axes[0].legend(handles, CLASSES, ncol=5, fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

## Event counts per class

In [ ]:
import collections
rows = []
for v in VIDEOS:
    det = collections.Counter(l for _, _, l in D[v]['events']); lab = collections.Counter(l for _, _, l in D[v]['labels'])
    for c in sorted(set(det) | set(lab)): rows.append((v, c, det[c], lab[c]))
print(f"{'video':8s} {'class':22s} {'detected':>8s} {'labels':>7s}")
for r in rows: print(f"{r[0]:8s} {r[1]:22s} {r[2]:8d} {r[3]:7d}")

## Scene analyses (images produced by `tools/`)
Motion heatmap, learned flow field, layout overlay, per-video alignment, queue-front heat versus the painted stop line, and the signal lamp strip.

In [ ]:
from IPython.display import Image, display
for name in ['C3896_motion_heatmap.jpg', 'flow_field.jpg', 'layout_overlay.jpg', 'align_C3902.jpg', 'queue_front_heat.jpg', 'light_strip_main.jpg']:
    print(name); display(Image(filename=str(ROOT / 'docs/assets' / name), width=900))